# P0: Simple models with Keras

**Goal**: implement **three models** for multiclass text classification on the [Women's E-commerce clothing reviews](https://github.com/ya-stack/Women-s-Ecommerce-Clothing-Reviews) dataset, two of them simple feed-forward models using a `Tokenizer` and `TextVectorizer`, respectively, and the third a Convolutional Neural Network (CNN) using a `TextVectorizer` layer and embeddings.

**Teams**: two person or three.

**Due date**: Check virtual campus.


### 1. Data preparation

The first step is to downlad the dataset (a `csv` file) from *GitHub*. Suggestions:
- You can use the utility function [`tensorflow.keras.utils.get_file()`](https://www.tensorflow.org/api_docs/python/tf/keras/utils/get_file) to download the file. You should set an absolute path to save the file, taking into account that, in *Google Colaboratory*, you have direct access to the folder `/content/`.
- There are many ways to load a `csv` in memory. One simple way is to use `csv.reader()`.

~~~
with open(path, newline='') as f:
    reader = csv.reader(f)
    data = list(reader)
~~~

The resulting data estructure (`data`) is a python list of lists (the reviews).

In [1]:
import csv

In [2]:
#خواندن فایل csv
with open('/home/jovyan/work/Women-s-Ecommerce-Clothing-Reviews-master/Womens Clothing E-Commerce Reviews.csv', newline='') as f:
    reader = csv.reader(f)
    data = list(reader)

print(data[0])
print(data[1])

['', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']
['0', '767', '33', '', 'Absolutely wonderful - silky and sexy and comfortable', '4', '1', '0', 'Initmates', 'Intimate', 'Intimates']


Once you have the rows of the `csv` file in a data structure (remember that the first one is the names of the attributes of the data set, and must be discarded) you have to preprocess the data for its use as an input to the neural networks:
1. Extract the textual data from the rows, included in the fields `Title` and `Review Text`, and join both fields if title is not empty.
2. Convert the field `Rating`, whose content are integers in the interval [1,5] into three classes: negative (ratings 1,2), neutral (rating 3) and positive (ratings 4,5).
3. The dataset contains about 23,000 reviwes. Reserve the first 18,000 for training, and the rest for validation.

In [3]:
#تمیز کردن داده ها به دو ستون ادغام تایتل و ریویو طبق درخواست تمرین
new_data = []

for row in data[1:]:
    if row[3] != "":
        new_data.append([row[3] + " " + row[4],row[5]])
    else:
        new_data.append([row[4],row[5]])
print(new_data[4])

['Flattering shirt This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!', '5']


In [4]:
#گروهبندی ریتینگ ها طبق درخواست تمرین
for row in new_data:
    if row[1] == '1' or row[1] == '2':
        row[1] = "negative"
    elif row[1] == '3':
        row[1] = "neutral"
    else:
        row[1] = "positive"
print(new_data[4])

['Flattering shirt This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!', 'positive']


In [5]:
#تقسیم داده ها به آموزش و تست طبق درخواست تمرین
training_data = new_data[:18000]
validation_data = new_data[18000:]

print(len(training_data))
print(len(validation_data))

18000
5486


### 2. Perceptron with Tokenizer.

In the first model, you are going to use a `Tokenizer()` object to process the training and validation texts, transforming each review into binary vectors (of length *n*, where *n* is the size of the vocabulary) in which the positions of the words appearing in the review will be coded as `1` (clue: you can use the method `texts_to_matrix()` for this). You can set a maximum size for the vocabulary (parameter `num_words`), but it is not necessary.

Remember that you have to use the `fit_on_texts()` method in order to build the vocabulary of the tokenizer from the training data.

In addition, you have to convert vectors with the labels (negative=0, neutral=1, positive=2) from the training and validation sets to a data type which make possible to use them with the loss function `categorical_crossentropy` (clue: you may want to use the utility function `tensorflow.keras.utils.to_categorical()`).

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np

I0000 00:00:1790545719.026909    5331 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [7]:
#توکنایز کردن داده های آموزش - یعنی استخراج کلمات از متون و ساخت وکبیولاری و اختصاص ایندکس به آنها و ذخیره در توکنایزر برای آماده سازی آموزش
tokenizer = Tokenizer()
#-------------------
#جدا سازی داده ها Xها از تارگت ها Yها برای داده های Train
training_texts = []

for row in training_data:
    training_texts.append(row[0])

print("----training_texts before tokenizing---")
print(training_texts[0])
print("---------------------------------------")

# Vocabulary را فقط از training یاد می‌گیریم تا اطلاعات validation وارد مرحله یادگیری نشود.
tokenizer.fit_on_texts(training_texts)
#--------------------
#جدا سازی داده ها Xها از تارگت ها Yها برای داده های Validation
validation_texts = []

for row in validation_data:
    validation_texts.append(row[0])
#--------------------
#حالا متن‌ها را با استفاده از آن vocabulary به بردارهای عددی تبدیل میکنم تا بتوانیم به شبکه عصبی برای آموزش آموزشی و مقایسه ارزیابی بدهیم.
x_train_1 = tokenizer.texts_to_matrix(training_texts, mode='binary')
x_val_1 = tokenizer.texts_to_matrix(validation_texts, mode='binary')

print("----training_texts after tokenizing---")
print(x_train_1)

----training_texts before tokenizing---
Absolutely wonderful - silky and sexy and comfortable
---------------------------------------
----training_texts after tokenizing---
[[0. 0. 0. ... 0. 0. 0.]
 [0. 1. 1. ... 0. 0. 0.]
 [0. 1. 1. ... 0. 0. 0.]
 ...
 [0. 1. 1. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 0.]]


In [8]:
#ذخیره لیبل های داده ها در جدول جداگانه برای هردو داده های آموزشی و ارزیابی
y_train_1 = []
for row in training_data:
    y_train_1.append(row[1])

print("------y_train before processing for one-hot encodeing------")
print(y_train_1[152])
print("------------------------------------")
#-----------------
y_val_1 = []
for row in validation_data:
    y_val_1.append(row[1])

print("------y_val before processing for one-hot encodeing------")
print(y_val_1[152])

------y_train before processing for one-hot encodeing------
neutral
------------------------------------
------y_val before processing for one-hot encodeing------
positive


In [9]:
#تبدیل لیبل ها به عددی برای تبدیل به one-hot encoding برای استفاده در لاس فانکشن categorical_crossentropy
y_train_num = []
for label in y_train_1:
    if label == 'positive':
        y_train_num.append(0)
    elif label == 'neutral':
        y_train_num.append(1)
    else:
        y_train_num.append(2)
        
print("------y_train in the process of one-hot encodeing------")
print(y_train_num[152])
print("------------------------------------")

#---------------
y_val_num = []
for label in y_val_1:
    if label == 'positive':
        y_val_num.append(0)
    elif label == 'neutral':
        y_val_num.append(1)
    else:
        y_val_num.append(2)

print("------y_val in the process of one-hot encodeing------")
print(y_train_num[152])
print("------------------------------------")

------y_train in the process of one-hot encodeing------
1
------------------------------------
------y_val in the process of one-hot encodeing------
1
------------------------------------


In [10]:
import tensorflow
#-----------------
#تبدیل به one-hot encoding داده های آموزشی برای پیشبینی شبکه های عصبی- و داده های ارزیابی برای مقایسه جوابشان با مقادیر پیشبینی شده
y_train_1= tensorflow.keras.utils.to_categorical(y_train_num)

print("------y_train after one-hot encodeing------")
print(y_train_1[152])
print("------------------------------------")

y_val_1= tensorflow.keras.utils.to_categorical(y_val_num)

print("------y_val after one-hot encodeing------")
print(y_train_1[152])
print("------------------------------------")

------y_train after one-hot encodeing------
[0. 1. 0.]
------------------------------------
------y_val after one-hot encodeing------
[0. 1. 0.]
------------------------------------


Now it is time to create the `Sequential` architecture of out first model. In this case, a simple perceptron with three layers (input, hidden, output) will suffice. A few pointers:
- You will need to set the `input_shape` of the first layer of the network to the size of the vocabulary in the `Tokenizer`.
- The number of units and the activation function in the output layer must be appropiate for a three-class classification problem.

In [11]:
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, Activation, Flatten

In [12]:
#معماری مدل رو برای آموزش میسازیم
model = Sequential([
    Input(shape=(13717,)),
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])

Now compile and train the model. You can use any optimizer you want, but the loss function must be `categorical_crossentropy`, the metric used will be `accuracy`, and you will provide the validation sets for the computation of the validation loss and validation accuracy at the end of each epoch of training, with the argument `validation_data`.

The model will train for 10 epochs.

Expect a validation accuracy of 0.80-0.83, approximately.

In [13]:
#مدل رو برای آموزش آماده میکنیم که از چه تکنیکی برای loss استفاده کنه و چطور وزن ها رو بهبود بده و بعد از محاسبه چه گزارشی را نشان دهد
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │       219,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 219,539 (857.57 KB)

 Trainable params: 219,539 (857.57 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
#مدل را آموزش میدهیم
model.fit(x_train_1, y_train_1, batch_size=32, epochs=10, verbose=1)

Epoch 1/10


W0000 00:00:1790545744.130049    5331 cpu_allocator_impl.cc:82] Allocation of 987624000 exceeds 10% of free system memory.


563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7791 - loss: 0.5118
Epoch 2/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8512 - loss: 0.3471
Epoch 3/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8923 - loss: 0.2721
Epoch 4/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9188 - loss: 0.2206
Epoch 5/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9403 - loss: 0.1773
Epoch 6/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9562 - loss: 0.1403
Epoch 7/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9682 - loss: 0.1098
Epoch 8/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9783 - loss: 0.0854
Epoch 9/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9840 - loss: 0.0672
Epoch 10/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9885 - loss: 0.0538


¿Does the validation accuracy grow with each epoch?

Yes, because with more epochs, the model has more time to learn the patterns in the training data, so its generalization may improve.


In [15]:
#مدل رو روی داده های ارزیابی، ارزیابی میکنه که چقدر درست عمل میکنه
loss,accuracy = model.evaluate(x_val_1, y_val_1, batch_size=16, verbose=0)
print("%s: %.2f%%" % (model.metrics_names[0], loss*100))
print("%s: %.2f%%" % (model.metrics_names[1], accuracy*100))

loss: 79.41%
compile_metrics: 81.24%


### 3. Perceptron with a TextVectorizer layer.

Now you are going to implement a new neural network, with two differences with respect to the previous one:
- We will use a `TextVectorizer` Layer instead of a `Tokenizer`.
- The loss function will be `sparse_categorical_crossentropy`.

Your first task is to set the `TextVectorization` layer. Remember you have to create the layer and call the method `adapt()` on the training data before adding the layer to the new model. You can use the default values when creating the layer if you wish, except for `output_mode` that has to be set to `'multi_hot'`, so a binary vector the size of the vocabulary is generated for each example, as `Tokenizer` did in the first model.

There is an important remark to consider when preparing string data for our models. With some recent updates in Keras/Tensorflow, string arrays are not converted into tensors properly anymore. So when you want to take string arrays as input for your training, you may need to convert explicitly to TF string tensors. A typical usage of `tf.convert_to_tensor()` would look like `x_train_tensor = tf.convert_to_tensor(x_train, dtype=tf.string)`.

In [16]:
from keras import layers

In [17]:
#TextVectorization متن‌ها را به sequenceهای عددی با طول ثابت 119 تبدیل کرده است و دیگه 13700 ستونی باینری نیست.
#TextVectorization می‌تواند خودش به‌عنوان یک Layer داخل شبکه عصبی قرار بگیرد.
#output_mode='multi_hot' برای تبدیل از ID کلمات به وکتوری با سایز ثابت vocabulary برای اینکه Dense بتونه وزن بده به کلمات
#متن خام رو به صورت ورودی میگیره و عملیات وکتوریزیشن روش انجام میشه

text_vectorizer = layers.TextVectorization(output_mode='multi_hot')
text_vectorizer.adapt(training_texts)

vectorized_text = text_vectorizer(training_texts)
print(vectorized_text)

W0000 00:00:1790545789.689422    5331 cpu_allocator_impl.cc:82] Allocation of 2509056000 exceeds 10% of free system memory.


tf.Tensor(
[[0 0 0 ... 0 0 0]
 [0 1 1 ... 0 0 0]
 [0 1 1 ... 0 0 0]
 ...
 [0 1 1 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 [0 0 0 ... 0 0 0]], shape=(18000, 17424), dtype=int64)


Now you can create the your second `Sequential` model, adding its layers one by one. Obviously, the previously created `TextVectorizer` goes first. Do not forget to define an input layer (have a look at the suggested input layer in the next exercise). You can add the rest of the layers after the text vectorizer.

In [18]:
from keras.models import Sequential
from keras.layers import Input,Dense

In [19]:
model = Sequential()
model.add(Input(shape=(), dtype=tensorflow.string))  #input must be one string at a time
model.add(text_vectorizer)
model.add(Dense(16, activation='relu'))
model.add(Dense(3, activation='softmax'))

Once the topology of the new model is set, you will set the datasets, compile and train it. Important:

- Remember that you are supposed to use `sparse_categorical_crossentropy`, so the label vectors for both training and validation will have to be of the appropiate type and dimensions.
- `TextVectorizer` will not accept its training (or validation) input as a list of strings. If you are using lists to store your input strings or lists of numpy arrays, remember to convert to tensors as pointed out above (`tf.convert_to_tensor()`).

You can use whichever optimizer you prefer, but you will use accuracy to measure the performance of the model, provide the validation data through the argument `validation_data`, and train for 10 epochs.

In [20]:
#مدل رو برای آموزش آماده میکنیم که از چه تکنیکی برای loss استفاده کنه و چطور وزن ها رو بهبود بده و بعد از محاسبه چه گزارشی را نشان دهد
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 17424)          │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │       278,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 278,851 (1.06 MB)

 Trainable params: 278,851 (1.06 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
#تبدیل متن خام به رشته قابل خواندن Tensorflow چون رشته رو باید داخل TextVectorization بذاریم اما از نوع Tensor.
x_train_2 = tensorflow.convert_to_tensor(training_texts, dtype=tensorflow.string)
x_val_2 = tensorflow.convert_to_tensor(validation_texts, dtype=tensorflow.string)
print(x_train_2[0])

tf.Tensor(b'Absolutely wonderful - silky and sexy and comfortable', shape=(), dtype=string)


In [22]:
y_train_sparse = tensorflow.convert_to_tensor(y_train_num, dtype=tensorflow.int32)
y_val_sparse = tensorflow.convert_to_tensor(y_val_num, dtype=tensorflow.int32)
print(y_train_sparse[0])

tf.Tensor(0, shape=(), dtype=int32)


In [23]:
#مدل رو آموزش میدیم
model.fit(x_train_2, y_train_sparse, batch_size=32, epochs=10, verbose=1, validation_data=(x_val_2, y_val_sparse))

Epoch 1/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7740 - loss: 0.5073 - val_accuracy: 0.7765 - val_loss: 0.4347
Epoch 2/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8511 - loss: 0.3511 - val_accuracy: 0.8338 - val_loss: 0.4073
Epoch 3/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8954 - loss: 0.2683 - val_accuracy: 0.8299 - val_loss: 0.4266
Epoch 4/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9248 - loss: 0.2093 - val_accuracy: 0.8265 - val_loss: 0.4593
Epoch 5/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9488 - loss: 0.1590 - val_accuracy: 0.8230 - val_loss: 0.5092
Epoch 6/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9657 - loss: 0.1189 - val_accuracy: 0.8183 - val_loss: 0.5622
Epoch 7/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9771 - loss: 0.0900 - val_accuracy: 0.8183 - val_loss: 0.6155
Epoch 8/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9847 - loss: 0.0678 - val_accu

In [24]:
#مدل رو روی داده های ارزیابی، ارزیابی میکنه که چقدر درست عمل میکنه
loss,accuracy = model.evaluate(x_val_2, y_val_sparse, batch_size=16, verbose=0)
print("%s: %.2f%%" % (model.metrics_names[0], loss*100))
print("%s: %.2f%%" % (model.metrics_names[1], accuracy*100))

loss: 76.32%
compile_metrics: 80.99%


¿Is the new model any better than the previous one?

No, but TextVectorizer was used because it provides a more modern and integrated text preprocessing pipeline, even though the Tokenizer model performed slightly better in this experiment.

### 4. CNN with TextVectorizer layer and word embeddings

Finally, you are going to train a third model with the following components:
- A `TextVectorizer` layer.
- An `Embedding` layer.
- One or more `Conv1D` layers.
- A `GlobalMaxPooling1D` layer.
- One or more `Dense` layers for the computation of results.
- A output layer with the appropiate activation function for a multiclass classifier.

You will use the functional API.

Our goal is to process the input texts token by token using a Convolutional Neural Network (CNN) and embeddings. The first step is to define the `TextVectorizer` layer. This time the output of this layer will be a vector of integer numbers (the input for the `Embedding` layer), with one integer for each token in the input text, so `output_mode` must be set to `int` or omitted (since `int` is the default value for this parameter). In addition, all sequences of integers (words) given to the embedding layer must have the same length. To ensure that, you will use the parameter `output_sequence_length` in the definition of the `TextVectorizer` (i.e. `output_sequence_length=100`). That will cut sequences longer than the value of `output_sequence_length` and pad shorter ones with zeros.

Once the layer is defined, it will be trained with the method `adapt()`.

In [25]:
# برخلاف multi_hot، در اینجا هر کلمه با ID خودش نمایش داده می‌شود،
# بنابراین اطلاعات ترتیب کلمات حفظ می‌شود و برای Embedding و CNN مناسب است.
text_vectorizer = layers.TextVectorization(output_mode='int', output_sequence_length = 100)
text_vectorizer.adapt(training_texts)

vectorized_text = text_vectorizer(training_texts)
print(vectorized_text)

tf.Tensor(
[[ 238  491  908 ...    0    0    0]
 [  20    8   17 ...    0    0    0]
 [ 156 1790  131 ...    2  454    6]
 ...
 [ 599    3   40 ...    8 3273    0]
 [1544   38  174 ...    0    0    0]
 [  27  174  779 ...    0    0    0]], shape=(18000, 100), dtype=int64)


You have to start the defintion of the model with an `Input` layer, e.g.:

~~~
inputs = keras.Input(shape=(1,),dtype=tf.string)
~~~

then you can add the `TextVectorizer`, `Conv1D`, ... layers.

The `Embedding` layer has at least two parameters: the size of the vocabulary and the size of the embeddings. For the vocabulary you have two choices: set it in avance when creating the layer, via de `max_tokens` parameter, or to let all tokens of the training set be part of the vocabulary. In the latter case, you can get the vocabulary size from the layer, using the method `vocabulary_size()`.

You must the set embedding dimension to a integer value, e.g. `30`.

In the `Conv1D` layer you have to set two parameters, `filters` and `kernel_size`. Both are integers. The first can have any integer value (e.g., `64`of `128`) but the higher is set, the bigger the number of computations will be, while the second should be small compared to the length of the sequences of words (e.g. `3` or `5`).

We finish with the output layer:

~~~
outputs = tf.keras.layers.Dense(...)(x)
~~~

where `x` is the output of the previous layer. At this point we can define the model:

~~~
model_functional = keras.Model(inputs=inputs, outputs=outputs)
~~~

In [26]:
#این اعداد فقط ID کلمات هستند؛
#اما CNN نمی‌خواهد فقط این IDها را ببیند. می‌خواهیم هر ID به یک بردار قابل یادگیری تبدیل شود.

#ایجاد کردن بردار Embedding برای یادگیری
embedding_layer = layers.Embedding(
    input_dim = text_vectorizer.vocabulary_size(), #تعداد کل کلمات موجود در vocabulary را می‌گیرد.
    output_dim = 30 #هر کلمه را به یک بردار ۳۰بعدی تبدیل می‌کند.
)

In [40]:
#اینجا هدف اینه که CNN از sequenceهای embedding شده، الگوهای محلی بین کلمات نزدیک به هم رو پیدا کنه.
#یادگیری کلمات نزدیک به هم
conv_layer = layers.Conv1D(
    filters = 64, #تعداد feature detectorهایی است که CNN یاد می‌گیرد. یعنی چند نوع الگوی مختلف را می‌خواهیم استخراج کنیم.
    kernel_size = 3 #مشخص می‌کند هر بار چند کلمه‌ی کنار هم را بررسی کند.
)

In [43]:
# لایه GlobalMaxPooling1D را ایجاد می‌کنیم تا از هر ویژگی استخراج‌شده توسط Conv1D، بیشترین مقدار را انتخاب کند
pooling_layer = layers.GlobalMaxPooling1D()

In [47]:
#لایه Dense میانی رو برای پردازش خروجی های استخراج شده میسازیم
dense_layer = layers.Dense(16, activation='relu')
#لایه Dense نهایی رو برای پیشبینی نهایی مدل میسازیم
output_layer = layers.Dense(3, activation='softmax')

In [48]:
#ورودی مدل را تعریف می‌کنیم؛ مدل قرار است متن خام دریافت کند
inputs = tensorflow.keras.Input(shape=(1,), dtype=tensorflow.string)
#متن خام را به اعداد تبدیل می‌کنیم
vectorized_text = text_vectorizer(inputs)
#اعداد را به بردارهای embedding تبدیل می‌کنیم
embedded_text = embedding_layer(vectorized_text)
# ویژگی‌های محلی متن را با CNN استخراج می‌کنیم
conv_output = conv_layer(embedded_text)
# مهم‌ترین ویژگی‌ها را از خروجی Conv1D انتخاب می‌کنیم
pooling_output = pooling_layer(conv_output)
#با لایه Dense خروجی های استخراج شده رو پردازش میکنیم
dense_output = dense_layer(pooling_output)
#حالا لایه Dense قبلی رو به این لایه نهایی وصل میکنیم
output = output_layer(dense_output)

In [49]:
#حالا کل زنجیره‌ای که تا الان ساختیم را به یک مدل واحد تبدیل می‌کنیم
#از اولین متغیر زنجیره به مدل وارد میشویم و تا آخرین متغیر زنجیره از مدل خارج میشویم
model_functional = tensorflow.keras.Model(
    inputs= inputs,
    outputs= output
)

You can use exactly the same datasets than in the previous model for training and validation, and the optimizer of you preference, but you will use accuracy as performance metric and sparse categorical crossentropy as loss function, provide the validation data through the argument `validation_data`, and train the model for 10 epochs.

In [52]:
#مدل رو برای آموزش آماده میکنیم که از چه تکنیکی برای loss استفاده کنه و چطور وزن ها رو بهبود بده و بعد از محاسبه چه گزارشی را نشان دهد
model_functional.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 17424)          │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │       278,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 278,851 (1.06 MB)

 Trainable params: 278,851 (1.06 MB)

 Non-trainable params: 0 (0.00 B)

In [53]:
#مدل رو آموزش میدیم
model_functional.fit(x_train_2, y_train_sparse, batch_size=32, epochs=10, verbose=1, validation_data=(x_val_2, y_val_sparse))

Epoch 1/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.7770 - loss: 0.5783 - val_accuracy: 0.7913 - val_loss: 0.4668
Epoch 2/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.8241 - loss: 0.4063 - val_accuracy: 0.8234 - val_loss: 0.4120
Epoch 3/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8606 - loss: 0.3292 - val_accuracy: 0.8257 - val_loss: 0.4042
Epoch 4/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.8968 - loss: 0.2618 - val_accuracy: 0.8336 - val_loss: 0.4300
Epoch 5/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9304 - loss: 0.1967 - val_accuracy: 0.8073 - val_loss: 0.4802
Epoch 6/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9570 - loss: 0.1373 - val_accuracy: 0.8161 - val_loss: 0.5455
Epoch 7/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9757 - loss: 0.0888 - val_accuracy: 0.8092 - val_loss: 0.6287
Epoch 8/10
563/563 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.9876 - loss: 0.0545 - val_acc

In [55]:
#مدل رو روی داده های ارزیابی، ارزیابی میکنه که چقدر درست عمل میکنه
loss,accuracy = model_functional.evaluate(x_val_2, y_val_sparse, batch_size=16, verbose=0)
print("%s: %.2f%%" % (model_functional.metrics_names[0], loss*100))
print("%s: %.2f%%" % (model_functional.metrics_names[1], accuracy*100))

loss: 85.29%
compile_metrics: 81.74%


¿Does the new model perform any better than the previous two?

It can be **yes based on Accuracy**, because it shows that the model correctly classified more samples. However, it can be **no based on Loss**, because it shows that some samples were classified correctly with low confidence.
